# YZ50 — Week 4: Neural Network Language Model

- A **neural network language model** predicts the next character by using a fixed **context window** of previous characters instead of relying on a simple bigram table.

- Each character is represented by a learnable **embedding vector**, and the embeddings of the previous three characters are combined and passed through an **MLP** to predict the next character.

- The model is trained using **minibatches**, **gradient descent**, and a **train / dev / test split** to measure how well it generalizes to unseen data.

- **Tanh saturation** can cause gradients to become very small when activations approach -1 or 1. Proper **weight initialization**, such as **Kaiming initialization**, helps keep activations and gradients in a useful range.

- **BatchNorm** normalizes hidden-layer activations during training, helping stabilize the network and improve the training process.

In this week, the simple counting-based language model is replaced with a **neural network language model**, while also exploring how **embeddings, initialization, activations, and BatchNorm** affect the learning process.

# Continuing to Build Makemore: Character-Level Language Model with MLP

---

For this setup, the learnable parameters are the **weights (W)**, **biases (b)**, and the **embedding lookup table (C)**. All of them are updated through backpropagation and gradient descent to minimize the loss.


# 1. Part 2 ile başla. Önceki üç harfi bağlam alan veri setini kur (X: 3 harf indeksi, Y: sıradaki harf). Embedding tablosunu (27x2) oluştur indeksleme ile embedding'leri çek.

In [76]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


In [77]:
# read in all the words
words = open('../data/names.txt', 'r').read().splitlines() # # Read the names from the file and split them into a list
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [78]:
len(words)

32033

## 1.1 Create the lookup tables

In [79]:
# build the vocabulary of characters and mappings to/from integers
# Create a lookup table to map characters to indices (0-25 for 'a'-'z', 26 for start/end tokens)
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


## 1.2 Construct the dataset

In [80]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], [] # X is the input data (context) for the neural network, Y is the target data (next character)
for w in words[:5]:

  print(w)
  context = [0] * block_size # initialize the context with zeros (start tokens)
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # update the context by removing the first character and adding the new character, this works like a sliding window
    #print('context:', context)

X = torch.tensor(X)
Y = torch.tensor(Y)

# For each word, it creates a context of length `block_size` (initialized with zeros) and iterates through each character in the word (plus a period to indicate the end of the word). For each character, it appends the current context to `X` and the index of the character to `Y`. After processing each character, it updates the context by removing the first character and adding the new character, effectively creating a sliding window of context for predicting the next character.

emma
... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
olivia
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
ava
... ---> a
..a ---> v
.av ---> a
ava ---> .
isabella
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
sophia
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [81]:
X.shape, X.dtype, Y.shape, Y.dtype

(torch.Size([32, 3]), torch.int64, torch.Size([32]), torch.int64)

## 1.3 Create the embeddings, embedding lookup table C

In [82]:
# Create the embedding table `C` with random values.
C = torch.randn((27,2)) # 27 rows for each character in the vocabulary (26 letters + 1 for the period), and 2 columns for the embedding dimension. Each row corresponds to a character's embedding vector.

In [83]:
C

tensor([[ 1.2016, -0.7739],
        [ 0.1165,  0.2011],
        [ 0.3218, -0.5923],
        [ 1.3292,  1.3971],
        [-2.0012, -0.0051],
        [-0.3274,  0.0734],
        [ 0.6243,  1.4156],
        [ 0.0518,  0.7786],
        [ 0.0652,  1.8565],
        [-0.5609, -0.0180],
        [ 0.3702, -0.8855],
        [-0.5950,  0.4438],
        [-1.5349, -0.3491],
        [ 1.2612,  0.1939],
        [-0.4015,  0.8630],
        [-0.9224,  1.5444],
        [-0.7901, -1.2808],
        [ 0.2959, -0.5328],
        [ 0.4793, -0.1020],
        [-0.7644,  1.8569],
        [-1.9917, -0.0907],
        [ 2.1524,  1.1051],
        [-0.8988, -0.8389],
        [ 1.3950,  1.0378],
        [-0.8572, -0.2674],
        [ 0.9836,  0.1322],
        [ 1.3417,  1.0259]])

In [84]:
C[5] # much faster than doing one-hot encoding and matrix multiplication, but we will do that next to show that it is equivalent

tensor([-0.3274,  0.0734])

In [85]:
F.one_hot(torch.tensor(5), num_classes=27).float() @ C # this is the same as C[5]

tensor([-0.3274,  0.0734])

In [86]:
C[torch.tensor([5, 0, 2])] # this gives us the embeddings for the characters with indices 5, 0, and 2

tensor([[-0.3274,  0.0734],
        [ 1.2016, -0.7739],
        [ 0.3218, -0.5923]])

In [ ]:
X[2] # this gives us the indices of the characters in the context of the 3rd training example (index 2)

tensor([ 0,  5, 13])

In [87]:
C[X[2]] # this gives us the embeddings for the characters in the context of the 3rd training example (index 2)

tensor([[ 1.2016, -0.7739],
        [-0.3274,  0.0734],
        [ 1.2612,  0.1939]])

In [88]:
C[X] # this is the same as doing one-hot encoding and matrix multiplication for all the indices in X

tensor([[[ 1.2016, -0.7739],
         [ 1.2016, -0.7739],
         [ 1.2016, -0.7739]],

        [[ 1.2016, -0.7739],
         [ 1.2016, -0.7739],
         [-0.3274,  0.0734]],

        [[ 1.2016, -0.7739],
         [-0.3274,  0.0734],
         [ 1.2612,  0.1939]],

        [[-0.3274,  0.0734],
         [ 1.2612,  0.1939],
         [ 1.2612,  0.1939]],

        [[ 1.2612,  0.1939],
         [ 1.2612,  0.1939],
         [ 0.1165,  0.2011]],

        [[ 1.2016, -0.7739],
         [ 1.2016, -0.7739],
         [ 1.2016, -0.7739]],

        [[ 1.2016, -0.7739],
         [ 1.2016, -0.7739],
         [-0.9224,  1.5444]],

        [[ 1.2016, -0.7739],
         [-0.9224,  1.5444],
         [-1.5349, -0.3491]],

        [[-0.9224,  1.5444],
         [-1.5349, -0.3491],
         [-0.5609, -0.0180]],

        [[-1.5349, -0.3491],
         [-0.5609, -0.0180],
         [-0.8988, -0.8389]],

        [[-0.5609, -0.0180],
         [-0.8988, -0.8389],
         [-0.5609, -0.0180]],

        [[-0.8988, -0

In [90]:
C[X].shape # (number of training examples, block_size, embedding dimension)

torch.Size([32, 3, 2])

In [91]:
emb = C[X]
emb.shape

torch.Size([32, 3, 2])

# 2. Gizli katmanı ve çıkış katmanını kur: embedding'leri düzleştir, W1 ve b1 ile tanh, W2 ve b2 ile logits. Loss'u geçen haftaki gibi elle hesapla, sonra F.cross_entropy ile aynı sonucu aldığını göster ve neden onu tercih ettiğimizi videodan anla.